# ML EDA: Portion Estimation Training Data

Exploring the synthetic training dataset to build intuition for the portion estimation problem.

**Core question**: Can we predict how many grams of food someone ate from a natural language description?

**Key insight to find**: The same container word ("bowl", "plate") maps to very different gram amounts depending on the food's physical properties (density class).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
    'figure.dpi': 120,
})
sns.set_palette('husl')

df = pd.read_csv('../ml/data/synthetic_training.csv')
print(f'Dataset: {len(df)} examples, {df["food_name"].nunique()} foods')
df.head(10)

## 1. Gram Distribution by Density Class

This is the single most important chart. Foods with the same description pattern produce wildly different gram amounts based on their physical density.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
order = df.groupby('density_class')['grams'].median().sort_values().index
sns.boxplot(data=df, x='density_class', y='grams', order=order, ax=axes[0])
axes[0].set_title('Gram Distribution by Density Class')
axes[0].set_xlabel('Density Class')
axes[0].set_ylabel('Grams')
axes[0].tick_params(axis='x', rotation=30)

# Violin plot — shows the shape better
sns.violinplot(data=df, x='density_class', y='grams', order=order, ax=axes[1], inner='quartile')
axes[1].set_title('Gram Distributions (violin)')
axes[1].set_xlabel('Density Class')
axes[1].set_ylabel('Grams')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Summary stats
print(df.groupby('density_class')['grams'].describe().round(0).to_string())

## 2. Why This Is Hard: Same Container, Different Grams

"A bowl of salad" and "a bowl of rice" use the same container word but the gram difference is ~3x. The model needs to learn that **density class × container → grams**.

In [ ]:
# Filter to descriptions containing 'bowl'
bowl_mask = df['description'].str.contains('bowl', case=False)
bowl_df = df[bowl_mask].copy()

if len(bowl_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Group by density class
    order = bowl_df.groupby('density_class')['grams'].median().sort_values().index
    sns.stripplot(data=bowl_df, x='density_class', y='grams', order=order,
                  alpha=0.6, jitter=True, ax=ax, size=5)
    sns.boxplot(data=bowl_df, x='density_class', y='grams', order=order,
                ax=ax, boxprops=dict(alpha=0.3), showfliers=False)
    
    ax.set_title('"A bowl of ..." — Same word, different grams')
    ax.set_xlabel('Density Class')
    ax.set_ylabel('Grams')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
    
    # Show specific examples
    print('\n--- Example "bowl" descriptions ---')
    for _, row in bowl_df.sample(min(10, len(bowl_df)), random_state=42).iterrows():
        print(f'  {row["description"]:45s} → {row["grams"]:4.0f}g  ({row["density_class"]})')
else:
    print('No bowl descriptions found')

## 3. Description Type Classification

Inferring description type from the text — this is a preview of the feature extraction we'll build next.

In [ ]:
import re

def classify_description(desc: str) -> str:
    """Classify a description into one of 4 types."""
    desc_lower = desc.lower().strip()
    
    # Metric: starts with or contains explicit gram/ml amounts
    if re.match(r'^(about\s+)?\d+\s*(g|ml)\b', desc_lower):
        return 'metric'
    
    # Count: starts with a number ("2 eggs", "3 slices")
    if re.match(r'^\d+\s', desc_lower) and not re.match(r'^\d+\s*(g|ml)\b', desc_lower):
        return 'count'
    
    # Container: contains container words
    container_words = ['bowl', 'plate', 'glass', 'cup', 'mug', 'pot',
                       'portion', 'piece', 'handful', 'dollop', 'side', 'bunch']
    if any(w in desc_lower for w in container_words):
        return 'container'
    
    return 'vague'


df['desc_type'] = df['description'].apply(classify_description)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count per type
type_counts = df['desc_type'].value_counts()
colors = {'vague': '#ff9500', 'container': '#007aff', 'metric': '#34c759', 'count': '#af52de'}
type_counts.plot.bar(ax=axes[0], color=[colors.get(t, '#999') for t in type_counts.index])
axes[0].set_title('Description Type Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Grams by type
type_order = ['metric', 'count', 'container', 'vague']
sns.boxplot(data=df, x='desc_type', y='grams', order=type_order, ax=axes[1],
            palette=colors)
axes[1].set_title('Gram Distribution by Description Type')
axes[1].set_xlabel('Description Type')
axes[1].set_ylabel('Grams')

plt.tight_layout()
plt.show()

print('\nDescription type breakdown:')
for dtype in type_order:
    subset = df[df['desc_type'] == dtype]
    pct = len(subset) / len(df) * 100
    print(f'  {dtype:12s}: {len(subset):4d} ({pct:5.1f}%)  mean={subset["grams"].mean():.0f}g  std={subset["grams"].std():.0f}g')

## 4. Calorie Density vs Portion Size

High-calorie-density foods (oils, nuts, chocolate) tend to have smaller portions. Low-calorie-density foods (salad, vegetables) have larger portions. This relationship is a strong feature for the model.

In [ ]:
# Load the reference data to get calories_per_100g for each food
ref = pd.read_csv('../ml/data/foods_reference.csv')
df_merged = df.merge(ref[['food_name', 'calories_per_100g']], on='food_name', 
                     how='left', suffixes=('', '_ref'))

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df_merged['calories_per_100g'], df_merged['grams'],
    c=df_merged['density_class'].map({
        'solid': '#007aff', 'granular': '#ff9500', 'liquid': '#34c759',
        'semisolid': '#ff3b30', 'leafy': '#5856d6', 'countable': '#af52de'
    }),
    alpha=0.3, s=15
)

ax.set_xlabel('Calorie Density (kcal/100g)')
ax.set_ylabel('Portion Size (grams)')
ax.set_title('Calorie Density vs Portion Size')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
    for l, c in [('solid', '#007aff'), ('granular', '#ff9500'), ('liquid', '#34c759'),
                  ('semisolid', '#ff3b30'), ('leafy', '#5856d6'), ('countable', '#af52de')]
]
ax.legend(handles=legend_elements, title='Density Class', loc='upper right')

plt.tight_layout()
plt.show()

# Correlation
corr = df_merged['calories_per_100g'].corr(df_merged['grams'])
print(f'\nCorrelation (calorie density vs grams): {corr:.3f}')
print('→ Negative correlation = higher calorie density → smaller portions')
print('  This makes physical sense and is a useful model feature.')

## 5. Size Modifier Effect

How much do words like "big", "small", "heaping" actually shift the gram prediction?

In [ ]:
def extract_size_modifier(desc: str) -> str:
    """Extract size modifier from description."""
    modifiers = ['tiny', 'small', 'medium', 'big', 'large', 'heaping', 'generous']
    desc_lower = desc.lower()
    for m in modifiers:
        if m in desc_lower:
            return m
    return 'none'

# Only look at container descriptions (where modifiers apply)
container_df = df[df['desc_type'] == 'container'].copy()
container_df['size_mod'] = container_df['description'].apply(extract_size_modifier)

# Normalise grams by food's typical serving to make comparison fair
container_df = container_df.merge(ref[['food_name', 'typical_serving_g']], on='food_name', how='left')
container_df['gram_ratio'] = container_df['grams'] / container_df['typical_serving_g']

mod_order = ['tiny', 'small', 'none', 'medium', 'big', 'large', 'heaping', 'generous']
existing_mods = [m for m in mod_order if m in container_df['size_mod'].values]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=container_df, x='size_mod', y='gram_ratio', order=existing_mods, ax=ax)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='1× typical serving')
ax.set_title('Size Modifier Effect on Portion Size')
ax.set_xlabel('Size Modifier')
ax.set_ylabel('Gram Ratio (vs typical serving)')
ax.legend()
plt.tight_layout()
plt.show()

print('Size modifier gram ratios (median):')
for m in existing_mods:
    subset = container_df[container_df['size_mod'] == m]
    print(f'  {m:12s}: {subset["gram_ratio"].median():.2f}x typical  (n={len(subset)})')

## 6. The Baseline Problem

How well does the simplest possible model do? If we just predict `typical_serving_g` for every description, what's the error?

In [ ]:
# Merge typical serving for baseline prediction
df_eval = df.merge(ref[['food_name', 'typical_serving_g']], on='food_name', how='left')

# Baseline: always predict typical_serving_g
df_eval['pred_baseline'] = df_eval['typical_serving_g']
df_eval['error_baseline'] = df_eval['pred_baseline'] - df_eval['grams']
df_eval['abs_error'] = df_eval['error_baseline'].abs()
df_eval['pct_error'] = (df_eval['abs_error'] / df_eval['grams'].clip(lower=1)) * 100

print('=== Baseline Model: Always Predict Typical Serving ===')
print(f'MAE (grams):     {df_eval["abs_error"].mean():.1f}g')
print(f'Median AE:       {df_eval["abs_error"].median():.1f}g')
print(f'MAPE:            {df_eval["pct_error"].mean():.1f}%')

within_20 = (df_eval['pct_error'] <= 20).mean() * 100
print(f'Accuracy@20%:    {within_20:.1f}%')
print()

# Breakdown by description type
print('Baseline error by description type:')
for dtype in ['metric', 'count', 'container', 'vague']:
    subset = df_eval[df_eval['desc_type'] == dtype]
    if len(subset) > 0:
        mae = subset['abs_error'].mean()
        acc20 = (subset['pct_error'] <= 20).mean() * 100
        print(f'  {dtype:12s}: MAE={mae:6.1f}g   Acc@20%={acc20:5.1f}%')

print()
print('→ The baseline is the FLOOR our model must beat.')
print('  Notice metric descriptions have LOW error (the number IS the answer)')
print('  and vague descriptions have HIGH error (no signal in the text).')
print('  The model\'s job: beat the baseline on container + vague descriptions.')

## 7. Category-Level Patterns

Do food categories show distinct portion patterns? This tells us if `category` is a useful feature.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

cat_order = df.groupby('category')['grams'].median().sort_values().index
sns.boxplot(data=df, x='category', y='grams', order=cat_order, ax=ax)
ax.set_title('Portion Size by Food Category')
ax.set_xlabel('Category')
ax.set_ylabel('Grams')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('Category summary:')
cat_stats = df.groupby('category').agg(
    n=('grams', 'count'),
    mean_g=('grams', 'mean'),
    median_g=('grams', 'median'),
    std_g=('grams', 'std'),
    mean_cal=('calories', 'mean'),
).round(0)
print(cat_stats.to_string())

## 8. Calorie Error Distribution

Ultimately, users care about calorie accuracy, not gram accuracy. Let's see how baseline gram errors translate to calorie errors.

In [ ]:
# Compute calorie error from gram error
df_eval = df_eval.merge(ref[['food_name', 'calories_per_100g']], on='food_name', 
                        how='left', suffixes=('', '_ref2'))
df_eval['cal_pred_baseline'] = df_eval['pred_baseline'] * df_eval['calories_per_100g'] / 100
df_eval['cal_error'] = (df_eval['cal_pred_baseline'] - df_eval['calories']).abs()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of calorie errors
axes[0].hist(df_eval['cal_error'], bins=50, edgecolor='white', alpha=0.7, color='#ff3b30')
axes[0].axvline(df_eval['cal_error'].mean(), color='black', linestyle='--', label=f'Mean: {df_eval["cal_error"].mean():.0f} kcal')
axes[0].axvline(df_eval['cal_error'].median(), color='blue', linestyle='--', label=f'Median: {df_eval["cal_error"].median():.0f} kcal')
axes[0].set_title('Baseline Calorie Error Distribution')
axes[0].set_xlabel('Absolute Calorie Error (kcal)')
axes[0].set_ylabel('Count')
axes[0].legend()

# By description type
type_order = ['metric', 'count', 'container', 'vague']
cal_errors = [df_eval[df_eval['desc_type'] == t]['cal_error'].values for t in type_order]
bp = axes[1].boxplot(cal_errors, labels=type_order, patch_artist=True)
type_colors = ['#34c759', '#af52de', '#007aff', '#ff9500']
for patch, color in zip(bp['boxes'], type_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1].set_title('Calorie Error by Description Type')
axes[1].set_ylabel('Absolute Calorie Error (kcal)')

plt.tight_layout()
plt.show()

print(f'Baseline mean calorie error: {df_eval["cal_error"].mean():.0f} kcal')
print(f'Baseline median calorie error: {df_eval["cal_error"].median():.0f} kcal')
print(f'\n→ This is how wrong the current system is. Our model\'s target: cut this in half.')

## Summary

**Key findings:**
1. Density class is the strongest signal — same container word produces 2-5x different gram amounts across food types
2. Vague descriptions ("some chicken", "had rice") are 43% of data and have the highest error — this is where the model adds most value
3. Size modifiers ("big", "small") create a ~2.5x range around the typical serving
4. Calorie density is negatively correlated with portion size — useful feature
5. The baseline (always predict typical serving) gives us the floor to beat

**Next step**: Build `ml/features.py` — extract structured features from descriptions that capture these patterns.